# Teach a little droid to walk
**163A / 263A · Robot kinematics and reinforcement learning**

Build a simulation, inspect the robot's joints, train a walking controller,
then change one reward and see how its gait changes.

**Today's questions:** What does the robot observe? What does its action mean?
How does a reward influence movement? How do we check whether it learned?

**Our route:** environment → kinematics → walking problem → SAC training
→ a short walking check → one reward experiment → drive your droid.

Examples are embedded in the text. Executable cells produce your own results.
Local timings are not Colab guarantees; we will print your actual steps and times.

## 1 · Meet the robot and create the environment

The real-world demonstration is Disney's BDX droid. Our classroom uses the
separately developed **BDX-R open-source model**, simulated with **MuJoCo Warp**
through mjlab. We train with **FastSAC**; this is not Disney's controller.


| Real-world inspiration | Our simulated walker |
|:---:|:---:|
| ![Disney BDX droid — D23 reference photograph](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-disney_bdx.png?v=572a2471b677) | ![Recorded instructor walking example](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/classroom_walking/baseline.gif?v=b1b2a5a0256a) |

Head gestures are procedural; the learned policy controls the legs.

Photo: [D23 / Disney](https://d23.com/bdx-droids-disneys-perfect-marriage-of-creativity-and-technological-achievement/), not relicensed. Simulation: the open-source BDX-R model with our recorded walking policy.

**Before running anything:** which joints do you expect to be most important
for moving a foot forward? Which might help keep the body upright?

Credits: [BDX-R model](https://github.com/BDX-R/BDX-R-MjLab),
[FastSAC project](https://younggyo.me/fastsac-humanoid/),
[Holosoma](https://github.com/amazon-far/holosoma), and
[mjlab](https://github.com/mujocolab/mjlab). Pinned revisions and license notices
are included with the lesson files.

### Install and configure the environment

Select **Runtime → Change runtime type → GPU**. Setup reports the assigned GPU.
Setup is separate from training.
An environment defines the robot, ground, actuators, observations, rewards,
and episode endings. We first inspect one robot, then train many copies.

In [ ]:
# @title Unpack the lesson files
import hashlib
import runpy
import subprocess
import sys
import tempfile
from pathlib import Path

# Download a small helper and verify it before running any downloaded code.
BUNDLE_URL = "https://github.com/TheNewRobot/bdxr-fastsac-classroom/releases/download/lesson-9fa67cec3f3f/BDXR_lesson_files.zip"
BUNDLE_SHA256 = "9fa67cec3f3fc38797195c085776e0dc2cf4037e2581a0c4cfc21fcf20491dac"
BOOTSTRAP_SHA256 = "9492e3ba32bed43816f4bf799bb8f60e7f3ab12fa24f30ff7457ff3a7e51e0fb"
WORK = (
    Path("/content/bdxr_classroom") if Path("/content").exists()
    else Path.cwd() / "bdxr_classroom_lab"
) / BUNDLE_SHA256[:12]
helper = Path(tempfile.mkdtemp(prefix="bdxr_setup_")) / "lesson_download.py"
fetch = "from urllib.request import urlretrieve; import sys; urlretrieve(*sys.argv[1:])"
subprocess.run(
    [sys.executable, "-c", fetch, BUNDLE_URL.rsplit("/", 1)[0] + "/lesson_download.py", str(helper)],
    check=True, timeout=60,
)
if hashlib.sha256(helper.read_bytes()).hexdigest() != BOOTSTRAP_SHA256:
    raise RuntimeError("Setup helper verification failed. Rerun this cell.")
runpy.run_path(str(helper))["prepare_lesson"](BUNDLE_URL, BUNDLE_SHA256, WORK)

In [ ]:
# @title Install pinned dependencies
if "WORK" not in globals() or not (WORK / "scripts/setup/setup.py").is_file():
    raise RuntimeError("Run the unpack cell first.")

from bdxr_classroom.lesson.setup import install_runtime

PYTHON, runtime_env = install_runtime(WORK)

In [ ]:
# @title Initialize the notebook's simulator connection
import json
import os
from bdxr_classroom.lesson.presentation import media_pair, read_json
from bdxr_classroom.lesson.runtime import Lesson
from bdxr_classroom.lesson.simulator import SimulatorCells

if "simulator" in globals():
    simulator.close()
simulator = SimulatorCells(PYTHON, WORK, runtime_env)
simulator.register(get_ipython())
lab = Lesson(WORK, PYTHON, runtime_env)
(WORK / "figures").mkdir(exist_ok=True)

### Define the walking task

These settings define the classroom walking task. Positive forward speed asks for
forward motion; negative speed asks for backward motion. Positive yaw rate
requests a left turn. Zero yaw requests no rotation, without a compass or
an external heading-correction controller.

The yaw- and lateral-bias weights discourage **sustained** turning error
and sideways drift. They do not add a heading-correction controller.

| Environment setting | Meaning |
|---|---|
| Physics step: 5 ms | Integrate forces and contacts |
| Control step: 20 ms | Ask the actor for new joint targets |
| Episode length: 10 s | Maximum duration before another practice reset |
| Speed range: −0.15 to +0.35 m/s | Reverse through brisk forward walking |
| Turn-rate range: ±0.45 rad/s | Right and left turns; zero requests straight motion |

`balanced` samples stopping, straight movement, turning, and combined commands.
Commands change every two seconds to practise stopping and restarting.
Small reset perturbations expose the learner to more than one starting posture.
These define the task; they are separate from SAC's network and replay settings.

In [ ]:
ENV_CONFIG = {
    "task_version": "v2",  # Ten leg-joint targets; the complete robot remains physical.
    "command_mode": "balanced",  # Practice forward/backward movement, stopping, and turns.
    "command_interval": 2.0,  # Seconds between training commands.
    "speed_min": -0.15,  # m/s, backward command limit.
    "speed_max": 0.35,  # m/s, forward command limit.
    "yaw_max": 0.45,  # rad/s, magnitude of the turning command.
    "yaw_tracking": "imu",  # Estimate turning rate from gyro and gravity direction.
    "yaw_std": 0.20,  # rad/s, width of the turning reward.
    "yaw_bias_weight": 10.0,  # Weight for reducing sustained turning error.
    "lateral_bias_weight": 2.0,  # Weight for reducing sustained sideways drift.
    "quiet_standing_weight": 3.0,  # Reward quiet legs and two grounded feet at zero command.
    "idle_velocity_weight": 0.5,  # Penalize leg fidgeting only at zero command.
    "contact_spin_weight": 2.0,  # Discourage twisting a planted foot instead of stepping.
    "initial_jitter": 0.03,  # Radians: practice small reset perturbations.
}
(WORK / "environment_settings.json").write_text(json.dumps(ENV_CONFIG, indent=2))

Cells marked `%%robot` run visible Python in the pinned simulator environment.
Variables persist between these cells. Ordinary cells display figures and
organize experiments in the notebook.

In [ ]:
%%robot
import json
from pathlib import Path
from bdxr_classroom.simulation.environment import make_env
from bdxr_classroom.lesson.simulator import setup_log

settings = json.loads(Path("environment_settings.json").read_text())
with setup_log("artifacts/logs/environment_setup.log"):
    env = make_env(num_envs=1, seed=7, render=True, **settings)
observations, _ = env.reset(seed=7)
print("Physics timestep:", env.cfg.sim.mujoco.timestep, "seconds")
print("Policy/control timestep:", env.step_dt, "seconds")
print("Episode time limit:", env.cfg.episode_length_s, "seconds")

In [ ]:
%%robot
from bdxr_classroom.lesson.kinematics import save_pose

save_pose(env, "figures/robot.png")

In [ ]:
from IPython.display import display, Image

display(Image(filename=str(WORK / "figures/robot.png"), width=720))

The simulator advances physics every **5 ms**; the policy chooses new targets
every **20 ms**. Training episodes last up to **10 s** and also end on the
configured fall conditions. A reset starts another attempt.

**Check your understanding:** why can physics need more updates than the policy?

## 2 · Explore the robot’s kinematics

The floating base describes the body's position and orientation; it is not a
motor. Each leg branches through **hip yaw → hip roll → hip pitch → knee → ankle**.
The neck/head is another branch. The tree below is read from the actual model;
fixed connections are collapsed to keep it readable.

### Instructor example · joints on the actual model

| Left-leg links and joint pivots | The same chain in the model |
|:---:|:---:|
| ![Left-leg joints on BDX-R](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-annotated_robot.png?v=fa3b952929f5) | ![Articulated kinematic tree](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-kinematic_tree.png?v=318ab7e68f24) |

The left leg faces the camera and is highlighted. Colored collars mark its
measured hinge centers; these are schematic overlays, not motor CAD.
Match their numbers to the tree. Other branches are faded or collapsed.

### Your activity · inspect joints, then move one

Run the inspection to read the actual joint axes and limits.
Then use the knee slider below to compare your prediction with forward kinematics.

In [ ]:
%%robot
from bdxr_classroom.lesson.kinematics import print_joint_table

print_joint_table(env)

### Joint angles determine foot position

For the slider, we view the **right leg** from its visible side. Let
$\mathbf q_R$ contain its five joint angles. Forward kinematics
computes the foot position relative to the torso:

$$\mathbf p_{\mathrm{foot}} = f(\mathbf q_R), \qquad
{}^{B}T_F = {}^{B}T_1(q_1)\,{}^{1}T_2(q_2)\cdots{}^{5}T_F.$$

Joint limits restrict which angles a motor can reach. The joint axis tells us
which direction it rotates about; the link geometry sets the distance to the
next joint. A hinge angle alone is therefore insufficient to locate the foot.
The torso frame provides a common origin for comparing foot positions.

Each transform contributes a joint rotation and a fixed link offset. The local
rotation axes and limits are listed above. Mirrored legs can use opposite axis
signs—do not assume that equal left/right angles produce identical motion.

**Predict:** holding all other angles fixed, what path will the foot trace as
the knee turns? Is the path a straight line?

The slider below moves the right knee with the root and other joints fixed.
Predict its foot path, then generate the interactive view to check your answer.

In [ ]:
%%robot
from bdxr_classroom.lesson.kinematics import sweep_joint

# Angles are in degrees here for readability; the model uses radians internally.
angles = range(-40, 21, 5)
foot_positions = sweep_joint(env, "Right_Knee", angles, "figures/knee_sweep.html")
print("Generated", len(foot_positions), "forward-kinematics configurations.")

In [ ]:
from bdxr_classroom.lesson.presentation import joint_explorer

explorer = joint_explorer(WORK / "figures/knee_sweep.html")

The slider changes one joint with the root and other joints fixed. The foot
coordinates are measured in the torso frame. This is **kinematics**, not a
demonstration that every pose is balanced. It traces one joint's motion,
not the robot's full reachable workspace.

Try a smaller angle interval or another right-leg joint within its listed limits.
Compare the geometric effect before changing any training reward.

## 3 · Define the walking problem

**Objective:** stay upright and follow the requested forward speed and turn rate.
Zero turn rate asks for approximately straight walking; commands can also request
stopping or reversing. The policy receives the robot state and these commands.
With no movement requested, aim for quiet standing with small balance corrections.
It outputs **ten leg-joint position targets**. Motor position controllers
generate forces to approach those targets; the policy does not directly place
the feet or teleport the robot.

$$q_i^{\mathrm{target}} = q_i^{\mathrm{default}} + s_i a_i$$

Here $a_i$ is a policy action and $s_i$ is its configured joint scale. This
connects the action vector to the joint-angle picture we just explored.

### Instructor example · the policy interface

For one walking robot, the **actor observation vector has length 47** and the
**critic observation vector has length 62**. These count scalar components,
not networks. The actor produces an **action vector of length 10**, converted
into leg-joint targets using the action scales.

The actor observes proprioception and commands. The critic's extra components
describe simulated motion and foot contact. These extra inputs are training-only.

### Your activity · verify the interface

Print the actual observation terms and action scales below. Use the table to
explain why true base linear velocity is not available to the deployed actor.

| Actor term | Scalars | Where it comes from |
|---|---:|---|
| Angular velocity | 3 | Gyroscope-like body rotation measurement |
| Gravity direction | 3 | Gravity projected into the body frame; an orientation estimate |
| Joint positions | 14 | Encoder-like angles relative to their joint references |
| Joint velocities | 14 | Joint velocities relative to their references |
| Previous actions | 10 | The controller's own previous leg commands |
| Movement command | 3 | Requested forward, sideways, and turning velocities |

**47 = 3 + 3 + 14 + 14 + 10 + 3.** The model measures 14 joints, but this
locomotion policy commands the ten leg joints. The sideways command is held
at zero in this lesson. Commands are instructions, not sensor measurements.

The critic adds 15 scalars: true linear velocity (3), two foot heights (2),
airtimes (2), contacts (2), and contact-force components (6). Thus its input
length is **62**. These simulator-only measurements help learning, but the
deployed actor cannot read them. In particular, it does not know world position
or heading and cannot secretly correct its path with a navigation controller.

In [ ]:
%%robot
print("Actor observation length per robot:", observations["actor"].shape[-1])
print("Critic observation length per robot:", observations["critic"].shape[-1])
print("Action vector length per robot:", env.action_manager.total_action_dim)
print("Parallel robots in this batch:", observations["actor"].shape[0])
for group in ("actor", "critic"):
    terms = env.observation_manager.active_terms[group]
    shapes = env.observation_manager.group_obs_term_dim[group]
    print(group, dict(zip(terms, shapes)))
print("Joint target scales:", env.cfg.actions["joint_pos"].scale)

| Quantity | Used by deployed actor? | Purpose |
|---|---|---|
| Gyroscope angular velocity; estimated gravity direction | Yes | Sense rotation and tilt |
| Joint encoders/velocity estimates; previous actions | Yes | Describe posture and recent motion |
| Forward-speed and yaw-rate commands | Yes | Specify the requested movement |
| True base linear velocity and contact information | No; critic/rewards may use them | Help training and evaluation |
| World position, absolute heading, terrain map | No | No hidden navigation controller |

**Actor:** chooses the next action. **Critic:** estimates how useful that action
may be during learning. Only the actor drives the deployed robot.

| Observations, actions and physics |
|:---:|
| ![Control loop: actor inputs, joint targets, physics, and rewards](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-control_loop.png?v=8462f0961d9f) |

### Define what good walking means

A reward combines several preferences:

$$r_t = w_v r_{\mathrm{speed}} + w_\omega r_{\mathrm{turn}}
      + r_{\mathrm{posture}} + r_{\mathrm{feet}} - c_{\mathrm{motion}}.$$

This is a grouped explanation; the next cell prints the actual terms and
weights. For example, an exponential tracking reward has the form
$r_{\mathrm{track}}=\exp(-e^2/\sigma^2)$: small tracking error earns more reward.
Simulator measurements may score the task without being given to the actor.

| Reward preference | What it encourages |
|---|---|
| Speed and turn tracking | Move according to the command |
| Upright posture | Keep the torso balanced |
| Foot airtime and clearance | Lift and place the feet during stepping |
| Quiet standing | Keep both feet supported when no movement is requested |
| Low planted-foot rotation | Turn by stepping instead of twisting planted feet |
| Small action changes | Avoid abrupt changes in motor targets |
| Low sustained yaw/sideways bias | Reduce persistent turning and sideways motion |

These preferences can compete: excessive smoothness can discourage useful
stepping, while chasing speed alone can produce unstable movement. Training
finds a compromise rather than executing a hand-written sequence of leg poses.

The next cell prints actual weights and parameters. These terms have different
units and scales; the largest weight need not make the largest contribution.

In [ ]:
%%robot
from prettytable import PrettyTable

rewards = PrettyTable(["Reward term", "Weight", "Parameters"])
for name, term in env.cfg.rewards.items():
    parameters = {
        key: value
        for key, value in term.params.items()
        if isinstance(value, (int, float, str, bool, tuple))
    }
    rewards.add_row([name, term.weight, str(parameters)])
print(rewards)

Speed/yaw rewards ask for command tracking. Posture and foot terms encourage
usable movement. Motion penalties discourage excessive effort or abrupt changes.
Later we will change one interpretable reward setting and inspect the result.
A higher return after changing weights does **not** automatically mean a better gait.

**Discuss:** why could rewarding only forward speed produce a poor walker?

### When does an episode end?

A task termination ends an unsuccessful attempt; a time limit ends a full
practice episode. The simulator then resets for another attempt. These endings
must be distinguished when learning how much future reward remains.
Here, a fall means torso tilt beyond 70° or base height below 0.12 m.
Inspect the actual rule parameters below; angles are stored in radians.

In [ ]:
%%robot
print("Episode time limit:", env.cfg.episode_length_s, "seconds")
for name, rule in env.cfg.terminations.items():
    print(name, "timeout:", rule.time_out, "parameters:", rule.params)

## 4 · Configure SAC and train

**SAC** learns from experience saved in a replay buffer. The actor proposes
actions; two critics estimate future return. Replay samples update the critics
and actor. An entropy term encourages
exploration rather than choosing the same action every time.

$$J(\pi)=\mathbb{E}\left[\sum_t\gamma^t
\left(r_t+\alpha\,\mathcal H(\pi(\cdot\mid o_t))\right)\right]$$

Read it as **future rewards + encouragement to explore**. $\gamma$ discounts
distant rewards; $\alpha$ controls the entropy contribution. FastSAC makes this
off-policy learning loop efficient on the GPU. This lesson uses SAC, not PPO.

| Learning from replay |
|:---:|
| ![SAC: collect experience, sample replay, update actor and critics](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-sac_loop.png?v=17e17f71fd24) |

### Separate learning settings from environment settings

During training, the actor samples exploratory actions and stores transitions
$(o_t,a_t,r_t,o_{t+1})$ in replay. Learning updates reuse these experiences;
they do not require a new simulated step for every gradient update. During
evaluation, the actor uses its deterministic action, without exploration noise.

The actor learns which actions to choose. The critics learn to predict return,
and their predictions supply a learning signal for the actor. A reward value
describes the current transition; a critic estimate includes expected future
rewards. Neither the reward calculation nor the critic is a deployed planner.

`num_envs` counts independent robots collecting experience. `batch_size` counts
sampled replay transitions; mirroring adds their reflected partners. These
are different quantities.
`steps` counts parallel simulation steps: with 512 robots, one step collects
512 transitions. The training curve plots collected transitions.

Keep the first-run settings below. We inspect one interaction, then train
a fresh policy. The seed identifies an experiment; judge the result using its
video and measurements rather than expecting identical GPU training.

In [ ]:
LEARNING_CONFIG = {
    "seed": 7,  # Seed for your baseline and matched reward experiment.
    "num_envs": 512,  # Independent simulations collecting experience.
    "batch_size": 2048,  # Replay samples before left/right augmentation.
    "buffer_size": 512,  # Replay slots per parallel robot.
    "steps": 12000,  # Parallel simulation steps.
    "minutes": 22,  # Time cap for 12,000 steps; setup/display time is separate.
    "gamma": 0.97,  # Discount applied to future reward.
    "updates": 4,  # Learning updates per simulation step.
    "actor_hidden": 256,  # Actor hidden-layer width.
    "critic_hidden": 384,  # Critic hidden-layer width.
    "mirror_replay": True,  # Reuse left/right mirrored experience.
    "compile": True,  # Compile the repeated GPU learning operations.
    "save_every": 3000,  # Keep periodic checkpoints; assess the final weights.
}
USE_WANDB = False  # @param {type:"boolean"}
LEARNING_CONFIG["wandb_mode"] = "online" if USE_WANDB else "disabled"

In [ ]:
# @title Optional W&B connection
if USE_WANDB and not (
    os.environ.get("WANDB_API_KEY") or Path.home().joinpath(".netrc").exists()
):
    import getpass

    lab.environment["WANDB_API_KEY"] = getpass.getpass(
        "W&B API key (not stored in the lesson): "
    )
print("W&B:", LEARNING_CONFIG["wandb_mode"])
print("Local metrics and checkpoints are saved even when W&B is disabled.")

Use a **GPU runtime** and allow **up to 22 minutes per training run**,
excluding setup and display. The target remains **12,000 steps**. The actual step count is
printed and saved; unfinished budgets must not be treated as matching experiments.

### Inspect the network and one interaction

Before the long training call, create a small inspection instance with the same
network widths. It uses one robot, eight replay slots, and no compilation or
mirroring solely to make the data flow easy to inspect. The full training
configuration above stays unchanged.

In [ ]:
(WORK / "learning_settings.json").write_text(json.dumps(LEARNING_CONFIG, indent=2))

In [ ]:
%%robot
from bdxr_classroom.learning.fastsac import FastSAC

inspection = json.loads(Path("learning_settings.json").read_text())
inspection.update(num_envs=1, buffer_size=8, compile=False, mirror_replay=False)
agent = FastSAC(
    observations["actor"].shape[-1],
    observations["critic"].shape[-1],
    env.action_manager.total_action_dim,
    inspection,
)
print("Actor inputs → hidden width → actions:",
      agent.dimensions[0], "→", inspection["actor_hidden"], "→", agent.dimensions[2])

In [ ]:
%%robot
# An untrained actor chooses an action; physics determines the result.
actions = agent.act(observations["actor"])
next_observations, reward, terminated, truncated, info = env.step(actions)
agent.add(observations, actions, next_observations, reward, terminated, truncated)
print("Action shape:", actions.shape)
print("Reward:", reward)
print("Replay write index after storing this transition:", agent.buffer.ptr)

This is one **collection step**, not a trained policy. The full training program
repeats collection, samples replay batches, and updates the critics and actor.
`terminated` marks a task termination; `truncated` marks a time limit. Keeping
the final observation lets the learner handle those endings correctly.

In [ ]:
%%robot
# Release the inspection environment before the full training jobs.
env.close()

In [ ]:
simulator.close()

### Train, then watch

Training starts from random networks. The helper saves the exact configuration,
final weights, resumable state, and real learning metrics in a timestamped folder.

In [ ]:
BASELINE = lab.train("command_walking_baseline", ENV_CONFIG, LEARNING_CONFIG)
print("Saved experiment:", BASELINE)

In [ ]:
from IPython.display import display, Video

baseline_video = lab.record(BASELINE, speed=0.30)
display(Video(str(baseline_video), embed=True, width=640))

In [ ]:
from IPython.display import display, Image

display(Image(filename=str(lab.learning_curve(BASELINE)), width=640))

Rising return suggests learning, but it does not establish good walking.
**Observe:** does the droid stay upright, move forward, and drift sideways?

## 5 · Briefly check the walking

Run **three 10-second forward trials** from slightly different starting poses.
We measure falls, speed error, turning error, and sideways drift. The numerical
tolerances stay unchanged; this short check does not test every command or terrain.
It targets under a minute of wall time and prints the actual duration.

World position measures drift for evaluation only—it never enters the actor.
A steady-looking video and a drift measurement answer different questions.

In [ ]:
from bdxr_classroom.lesson.brief import assess_walking

BASELINE_CHECK = assess_walking(lab, BASELINE)

**Interpret:** “stayed upright” and “followed the command” are distinct results.
Record shortcomings honestly. This cell does not select another checkpoint,
retrain the robot, or launch more checks automatically.

## 6 · Change one reward and compare

Predict what happens when we reward flatter feet. Change **one scalar reward
weight**, then train a new policy from scratch. Your first run is the unchanged
control: keep the seed, commands, and learning budget identical.

| Instructor example at equal commands |
|:---:|
| ![Fresh baseline and flatter-foot comparison, with check outcomes](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/classroom_walking/reward_comparison.gif?v=77ad24939e9c) |

The preview loops automatically; it has no Play button.
If it stays still, [open the comparison video](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/classroom_walking/reward_comparison.mp4?v=2c7ed5d4c769)
and press ▶ in the video player.

Only `level_feet_weight` changes: **0 → 0.75**. Both trained from scratch for
12,000 steps. Measured swing-foot tilt fell from **9.7° to 5.2°**, while foot
landings increased from **4.11 to 4.52 per second**. Both passed the brief
walking check.

**Watch the feet:** the change is subtle—flatter feet and slightly quicker
steps. This example shows how one reward influences movement, rather than a
dramatically different gait. Your measurements may differ.

In [ ]:
PREDICTION = "The feet will stay flatter during stepping."  # @param {type:"string"}
LEVEL_FEET_WEIGHT = 0.75  # @param {type:"number"}
VARIANT_ENV = {**ENV_CONFIG, "level_feet_weight": LEVEL_FEET_WEIGHT}
BASELINE_STEPS = read_json(BASELINE / "metrics/timing.json")["step"]
VARIANT_LEARNING = {**LEARNING_CONFIG, "steps": BASELINE_STEPS}
print("My prediction:", PREDICTION)

In [ ]:
# No parent checkpoint: the variant starts from random networks too.
VARIATION = lab.train("level_feet", VARIANT_ENV, VARIANT_LEARNING)
variation_video = lab.record(VARIATION, speed=0.30)

In [ ]:
media_pair([
    ("Your baseline", baseline_video, ""),
    ("Your reward variation", variation_video, ""),
])

In [ ]:
VARIATION_CHECK = assess_walking(lab, VARIATION)

In [ ]:
from bdxr_classroom.lesson.gait import compare_gaits

compare_gaits(lab, BASELINE, VARIATION,
              assessments=[BASELINE_CHECK, VARIATION_CHECK])

**Explain:** did foot tilt change as predicted, while walking remained useful?
The variant targets the baseline's actual completed step count, including when
the first run hit its time limit. Check that both runs finished that budget.
Changing a reward changes
the return scale, so a larger return alone does not establish improvement.

## 7 · Drive your trained robot and save

Finish by driving **your own walking controller** through Desert Droid Run.
Steer between the markers, slow down for narrow passages and uneven ground,
then reach the raised finish. Required obstacles use walking; no hopper is needed.
You command speed and turn rate. The learned actor supplies the leg actions.

| The walking course |
|:---:|
| ![Desert walking course](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/course_preview.png?v=fc90d6eab59f) |

Click **Resume**, then hold a direction button or **W/S** to move and **A/D**
to turn. Release to request stopping. **Slow / Normal / Fast** select speed;
they do not start movement. Actual motion and requested commands are shown
separately. Falling keeps accumulated time and falls; Pause freezes simulation.
On remote VS Code, forward port 8765 if the panel is blank.

**Choose a checked policy:** `auto` uses a passing run, preferring baseline if
both pass. The summary reuses your existing checks—no extra training or
evaluation. A failed selection will not launch the game. Passing the short
forward check does not guarantee successful turning or obstacle traversal.

In [ ]:
from bdxr_classroom.lesson.policy_choice import choose_walking_policy

POLICY = "auto"  # @param ["auto", "baseline", "variation"]
walking = choose_walking_policy(
    {"baseline": BASELINE, "variation": VARIATION},
    {"baseline": BASELINE_CHECK, "variation": VARIATION_CHECK},
    preference=POLICY,
)
SELECTED_WALK_CHECKPOINT = walking

In [ ]:
from bdxr_classroom.lesson.play import launch_game, display_game

GAME_PORT = 8765
if walking:
    status = launch_game(lab, walking, port=GAME_PORT, course="configs/walking_course.json")
    display_game(GAME_PORT)
else:
    print("Choose a policy with a passing check in the preceding cell.")

### Optional · watch and keep your result

To keep a clip, open **Recording · optional** in the game and start recording.
Enable **WATCH_RECORDING** below to pause and watch your latest clip.
Saving is optional: the ZIP contains configurations, measurements,
checkpoints, and gameplay. Include replay only for a larger resumable backup.

In [ ]:
from bdxr_classroom.lesson.play import optional_recording

WATCH_RECORDING = False  # @param {type:"boolean"}
replay = optional_recording(lab, GAME_PORT, WATCH_RECORDING)

In [ ]:
from bdxr_classroom.lesson.exports import optional_download

SAVE_RESULTS = False  # @param {type:"boolean"}
INCLUDE_REPLAY = False  # @param {type:"boolean"}
backup = optional_download(WORK, lab.runs, SAVE_RESULTS, INCLUDE_REPLAY)

**Submit:** your walking video, short check, and reward prediction/comparison.
State actual steps/timings and one limitation. A gameplay clip is optional.

After class: [the separate hopping extension](https://colab.research.google.com/github/TheNewRobot/bdxr-fastsac-classroom/blob/main/notebooks/BDXR_Hopping_Extra.ipynb)
explores another behavior. It is not required to finish this lesson or drive the course.